# Text Tokenisation

**Goal:** Convert words into numbers for mathematical interpretation by algorithms.

## Text Tokenization

### 1. What is it?
Text tokenization is the process of breaking down continuous text into smaller pieces called tokens. It is the mandatory first step an LLM takes to convert raw human language into a format it can mathematically process.

### 2. Why do we use it?
Computers cannot read text as a whole string. By breaking sentences into standardized units (like words or subwords), tokenization transforms unstructured text into a predictable sequence of distinct building blocks that can be mapped to numerical IDs.

### 3. How does it work?
1. **Lookup:** The tokenizer checks a pre-built dictionary (vocabulary) containing tens of thousands of common characters, words, and word fragments.
2. **Match and Split:** It scans the text and breaks it down into the largest fragments it recognizes from its dictionary (e.g., "unhappily" becomes `["un", "happi", "ly"]`).
3. **Convert to IDs:** It replaces every fragment with its corresponding unique number ID from the dictionary.

### 4. The Logic
Modern tokenizers use **Subword Tokenization** (like Byte-Pair Encoding) to handle missing words. The mathematical objective is to represent text using the minimum number of tokens without creating "unknown" words:

$$\text{Optimal Splitting} = \arg\max \prod_{t \in T} P(t)$$

* **$T$:** The resulting sequence of tokens generated from the input text.
* **$t$:** A single token within that sequence.
* **$P(t)$:** The frequency or probability of that token appearing in the training corpus.
* **The Product and Optimization ($\arg\max \prod$):** The algorithm picks the combination of splits that maximizes the overall probability, prioritizing common full words over fragmented pieces whenever possible.

#### Glossary
* **Token**: A single unit of text (a word, subword, or punctuation mark) produced by the tokenizer.
* **Vocabulary (Vocab)**: The fixed master list of all unique tokens that a specific LLM is trained to recognize.
* **Token ID**: The unique integer assigned to each token in the vocabulary, used as direct input for word embeddings.

## Code Example
Converting words to tokens and token IDs using the tiktoken module.

**Note:** This is a very simple example. To see how these pieces connect, see the `examples` folder in the root.

In [2]:
import tiktoken

# Load the gpt-4 tokenizer (Local)
enc = tiktoken.get_encoding("cl100k_base")

text = "The dog barked unhappily."

# 1. Convert text to Token IDs
token_ids = enc.encode(text)

# 2. Convert Token IDs back to byte tokens to see the splits
tokens = [enc.decode_single_token_bytes(i).decode('utf-8', errors='replace') for i in token_ids]

print(f"Original Text: {text}\n")
print(f"1. Text Tokens: {tokens}")
print(f"2. Token IDs:   {token_ids}")

Original Text: The dog barked unhappily.

1. Text Tokens: ['The', ' dog', ' b', 'arked', ' unh', 'app', 'ily', '.']
2. Token IDs:   [791, 5679, 293, 43161, 31175, 680, 1570, 13]


## Special Tokens

**Adding special tokens** is the process of inserting hidden, non-text markers into a tokenized sequence to give structural metadata to the LLM.

Common examples include:

* `[BOS]` or `<s>`: Beginning of Sequence (signals where text starts)
* `[EOS]` or `</s>`: End of Sequence (tells the model to stop generating text)
* `[PAD]`: Padding (fills empty space so sentences match a fixed mathematical size)
* `[SEP]`: Separator (splits two distinct texts, like a Question and an Answer)

---

### Why It Is Required

1. **Context and Boundaries:** Without special tokens, an LLM cannot tell where one document ends and a completely unrelated one begins. An `[EOS]` token prevents the model from bleeding separate concepts together.
2. **Fixed-Batch Math:** Machine learning models process data in rigid, uniformly sized grids (matrices). If Sentence A has 5 words and Sentence B has 10 words, the tokenizer adds `[PAD]` tokens to Sentence A so both vectors match the exact same mathematical length.
3. **Task Routing:** Special structural tokens tell the network how to behave, signaling whether it is reading a conversational prompt, a system instruction, or a passage to be translated.

## Byte Pair Encoding

**Byte Pair Encoding (BPE)** is a data-driven subword tokenization algorithm used by modern LLMs (like GPT-4 and Llama). Instead of breaking text into whole words or individual characters, BPE dynamically builds a vocabulary of the most frequent letter combinations found in a text corpus.

### How it Works (The Core Logic)

1. **Initialize with Characters:** The algorithm starts by treating every individual character (letters, punctuation, and spaces) as a baseline token.
2. **Count Adjacent Pairs:** It scans the entire training text and counts how frequently pairs of tokens appear next to each other (e.g., how often `d` is followed by `e`, or `t` by `h`).
3. **Merge the Most Frequent Pair:** The most common pair is merged into a brand new single token. For example, if `t` and `h` appear together the most, they become the token `th`.
4. **Repeat:** The process repeats iteratively for a fixed number of rounds (e.g., 32,000 or 50,000 times). Over time, common pairs turn into syllables (`the`), which then turn into full words (`there`), while rare words remain broken down into sub-components.

### Why it is Used

* **No "Unknown" Words:** Because it keeps individual characters as a fallback, it can break down any unseen word or typo into recognized character pieces, meaning it never fails to read a word.
* **Balanced Vocabulary Size:** It allows the LLM to efficiently process common words as a single token while cleanly compressing rare or complex words into small combinations of subword tokens.

## Resources
* [Visualise tokens in a sample text](https://gpt-tokenizer.dev/)